# 02 Data Cleaning & Preprocessing

## Objective
Clean and preprocess the raw pharmacy datasets directly inside this notebook.

This notebook:
- loads the raw CSV files
- handles missing values
- removes duplicate rows
- converts date columns
- clips numeric outliers using the IQR rule
- optionally saves cleaned datasets for later notebooks


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype

DATE_COLUMNS = {
    'inventory': ['date', 'expiry_date'],
    'sales': ['date'],
    'prescriptions': ['date'],
    'employees': []
}

candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = next(
    (path for path in candidate_roots if (path / 'data').exists()),
    None
)

if project_root is None:
    raise FileNotFoundError(
        'Could not locate the project data folder. Run this notebook from the project root or notebooks folder.'
    )

data_dir = project_root / 'data'
cleaned_dir = data_dir / 'cleaned'

print(f'Project Root: {project_root}')
print(f'Data Directory: {data_dir}')


Project Root: D:\projects\ai-ml-projects\PharmaEase_correct\pharma_ease_ai
Data Directory: D:\projects\ai-ml-projects\PharmaEase_correct\pharma_ease_ai\data


In [2]:
def load_dataset(filename: str) -> pd.DataFrame:
    file_path = data_dir / filename
    if not file_path.exists():
        raise FileNotFoundError(f'Dataset not found: {file_path}')
    return pd.read_csv(file_path)


def load_all_datasets() -> dict[str, pd.DataFrame]:
    return {
        'inventory': load_dataset('inventory.csv'),
        'sales': load_dataset('sales.csv'),
        'prescriptions': load_dataset('prescriptions.csv'),
        'employees': load_dataset('employees.csv')
    }


def clean_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.copy()
    for column in clean_df.columns:
        if is_numeric_dtype(clean_df[column]):
            clean_df[column] = clean_df[column].fillna(clean_df[column].median())
        else:
            clean_df[column] = clean_df[column].fillna('unknown')
    return clean_df


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates().reset_index(drop=True)


def convert_dates(df: pd.DataFrame, date_columns: list[str]) -> pd.DataFrame:
    date_df = df.copy()
    for column in date_columns:
        if column in date_df.columns:
            date_df[column] = pd.to_datetime(date_df[column], errors='coerce')
    return date_df


def handle_outliers_iqr(df: pd.DataFrame, multiplier: float = 1.5) -> pd.DataFrame:
    out_df = df.copy()
    numeric_cols = out_df.select_dtypes(include=[np.number]).columns

    for column in numeric_cols:
        q1 = out_df[column].quantile(0.25)
        q3 = out_df[column].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - multiplier * iqr
        upper = q3 + multiplier * iqr
        out_df[column] = out_df[column].clip(lower=lower, upper=upper)

    return out_df


def preprocess_dataset(name: str, df: pd.DataFrame) -> pd.DataFrame:
    processed = clean_missing_values(df)
    processed = remove_duplicates(processed)
    processed = convert_dates(processed, DATE_COLUMNS.get(name, []))
    processed = handle_outliers_iqr(processed)
    return processed


In [3]:
raw_data = load_all_datasets()

print('Raw Data Shapes:')
print('=' * 40)
for name, df in raw_data.items():
    print(f'{name:15} | {df.shape}')


Raw Data Shapes:
inventory       | (1500, 5)
sales           | (2000, 6)
prescriptions   | (1300, 4)
employees       | (1200, 5)


In [4]:
quality_before = []

for name, df in raw_data.items():
    quality_before.append({
        'dataset': name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'missing_values': int(df.isna().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum())
    })

quality_before_df = pd.DataFrame(quality_before)
quality_before_df


,dataset,rows,columns,missing_values,duplicate_rows
0,inventory,1500,5,0,0
1,sales,2000,6,0,0
2,prescriptions,1300,4,0,0
3,employees,1200,5,0,0


In [5]:
cleaned_data = {
    name: preprocess_dataset(name, df)
    for name, df in raw_data.items()
}

print('Cleaned Data Shapes:')
print('=' * 40)
for name, df in cleaned_data.items():
    print(f'{name:15} | {df.shape}')


Cleaned Data Shapes:
inventory       | (1500, 5)
sales           | (2000, 6)
prescriptions   | (1300, 4)
employees       | (1200, 5)


In [6]:
quality_after = []

for name, df in cleaned_data.items():
    quality_after.append({
        'dataset': name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'missing_values': int(df.isna().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum())
    })

quality_after_df = pd.DataFrame(quality_after)
quality_report = quality_before_df.merge(
    quality_after_df,
    on='dataset',
    suffixes=('_before', '_after')
)
quality_report


,dataset,rows_before,columns_before,missing_values_before,duplicate_rows_before,rows_after,columns_after,missing_values_after,duplicate_rows_after
0,inventory,1500,5,0,0,1500,5,0,0
1,sales,2000,6,0,0,2000,6,0,0
2,prescriptions,1300,4,0,0,1300,4,0,0
3,employees,1200,5,0,0,1200,5,0,0


In [7]:
for name, df in cleaned_data.items():
    print(f'\n{name.upper()} SAMPLE')
    display(df.head())



INVENTORY SAMPLE


,drug_name,quantity,usage_rate,date,expiry_date
0,Aspirin,274,12.48,2023-01-01,2023-11-29
1,Paracetamol,169,7.34,2023-01-01,2023-11-12
2,Amoxicillin,80,3.26,2023-01-01,2025-10-28
3,Paracetamol,410,17.96,2023-01-03,2025-11-14
4,Losartan,436,18.47,2023-01-04,2025-06-25



SALES SAMPLE


,transaction_id,items,quantity,price,customer_id,date
0,TX-100833,Respiratory Care,4,164.82,CUST-2261,2023-01-02
1,TX-100668,Respiratory Care,2,75.86,CUST-2461,2023-01-02
2,TX-101300,Pain Relief,3,91.64,CUST-1151,2023-01-02
3,TX-101596,Allergy Care,7,120.64,CUST-1197,2023-01-04
4,TX-100081,Allergy Care,4,174.53,CUST-2437,2023-01-04



PRESCRIPTIONS SAMPLE


,patient_id,symptoms,drug_prescribed,date
0,PAT-81636,fever headache,Paracetamol,2023-01-01
1,PAT-88964,respiratory infection cough,Azithromycin,2023-01-02
2,PAT-67983,fever headache,Paracetamol,2023-01-03
3,PAT-27009,high cholesterol chest discomfort,Atorvastatin,2023-01-04
4,PAT-47576,mild pain blood thinning,Aspirin,2023-01-07



EMPLOYEES SAMPLE


,employee_id,shifts,performance_score,attendance,workload
0,EMP-1000,Night,95.67,97.76,33.38
1,EMP-1001,Night,84.12,77.43,46.13
2,EMP-1002,Morning,94.01,95.66,40.77
3,EMP-1003,Morning,86.56,97.61,35.22
4,EMP-1004,Morning,60.78,97.06,26.84


In [8]:
cleaned_dir.mkdir(parents=True, exist_ok=True)

for name, df in cleaned_data.items():
    output_path = cleaned_dir / f'{name}_cleaned.csv'
    df.to_csv(output_path, index=False)
    print(f'Saved: {output_path.name}')


Saved: inventory_cleaned.csv
Saved: sales_cleaned.csv
Saved: prescriptions_cleaned.csv
Saved: employees_cleaned.csv


## Summary

- All datasets are cleaned inside this notebook.
- Missing values are imputed using median for numeric columns and `unknown` for text columns.
- Duplicate rows are removed.
- Date columns are converted to datetime where available.
- Numeric outliers are clipped with the IQR method.
- Cleaned files are saved to `data/cleaned/` for the next notebooks.
